In [1]:
# Import all necessary packages
import pandas as pd
from scipy import stats
import matplotlib.pyplot as plt
import seaborn as sns
import rootutils

# Set directories and load data
path_root = str(rootutils.find_root(indicator=".project-root"))
path_plots = f"{path_root}/plots"
path_data = f"{path_root}/data/cytokines-regression"
df_feats = pd.read_excel(f"{path_data}/features.xlsx")
imms = df_feats.columns.to_list()
df = pd.read_excel(f"{path_data}/data.xlsx")

# Create dataframe for the figure
df_epi_imm_corr = pd.DataFrame(index=imms, columns=list(range(1, 101)))
for imm in imms:
    corrs = []
    for cpg in df_feats[imm]:
        res = stats.pearsonr(df[imm], df[cpg], alternative='two-sided')
        corrs.append(abs(res.statistic))
    df_epi_imm_corr.loc[imm, :] = sorted(corrs, reverse=True)

# Plot Figure 2a  
df_fig = df_epi_imm_corr.astype(float)
df_fig = df_fig.T
sns.set_theme(style='ticks', font_scale=1.0)
fig, ax = plt.subplots(figsize=(10, 30))
heatmap = sns.heatmap(
    df_fig,
    annot=False,
    cmap='hot',
    linewidth=0.1,
    linecolor='black',
    cbar_kws={
        'orientation': 'horizontal',
        'location': 'top',
        'fraction': 0.05,
        'pad': 0.038,
        'aspect': 60
    },
    annot_kws={"size": 12},
    ax=ax
)
ax.set_xlabel('')
ax.set_ylabel('Top CpGs')
ax.tick_params(axis='y', rotation=0)
ax.tick_params(axis='x', top=True, labeltop=True, bottom=False, labelbottom=False, rotation=90)
heatmap_pos = heatmap.get_position()
ax.figure.axes[-1].set_title(fr"|Pearson's $R$|", fontsize='x-large')
ax.figure.axes[-1].tick_params(labelsize='large')
for spine in ax.figure.axes[-1].spines.values():
    spine.set_linewidth(1)
plt.savefig(f"{path_plots}/figure2a.png", bbox_inches='tight', dpi=200)
plt.savefig(f"{path_plots}/figure2a.pdf", bbox_inches='tight')
plt.close(fig)